In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import cv2


# Generate normal dataset

In [ ]:
# Load an HDF5 dataset (replace 'filename.h5' with your file)

file_name="2segments_smooth_input_rand_compressed"
file_name="1segment_step_input_rand_compressed"

file_path = f'data/scr_match2/{file_name}.h5' 


res=542
y_offset=24
downscale_res=32
cut_off= 120


if "2seg" in file_name: 
    p_indexes=[0,2,3,5]
else:
    p_indexes=[0,2]

n_read=1000
with h5py.File(file_path, 'r') as hdf5_file:
    # Print all the root-level groups/datasets for inspection
    print("Keys in HDF5 file:", list(hdf5_file.keys()))
    trajectory_time=hdf5_file['trajectory_time'][:]

    trajectory_start=np.nonzero(trajectory_time)[0][-0]
    trajectory_end=np.nonzero(trajectory_time)[0][-1]+1

    # Process each image: extract, downscale, grayscale, rotate
    images_list = []
    for i in range(trajectory_end):
        if i % 100 == 0:
            print(f"Processing image {i+1}/{trajectory_end}")
        img_rgb = hdf5_file['images'][i, :res, y_offset:y_offset+res]

        if "1seg" in file_name:
            img_rgb_zoomed = img_rgb[cut_off: -cut_off, 0: -2*cut_off]
        else:
            img_rgb_zoomed = img_rgb

        img_rgb_downscaled = cv2.resize(img_rgb_zoomed, (downscale_res, downscale_res), interpolation=cv2.INTER_AREA)
        img_gray = cv2.cvtColor(img_rgb_downscaled, cv2.COLOR_RGB2GRAY)
        image_processed = np.rot90(img_gray, k=-1)
        images_list.append(image_processed)
    images = np.stack(images_list, axis=0)
    pressure=hdf5_file['pressure'][:trajectory_end,p_indexes]/1e5
    trajectory_time_used=trajectory_time[:trajectory_end]

    #image_timestamps=hdf5_file['image_timestamps'][:]

    for key in hdf5_file.keys():
        print(key)
        print(hdf5_file[key].shape)
        print(len(hdf5_file[key]))

In [ ]:
# Save results to npz file
if "2seg" in file_name:
    npz_dict = {'images': images, 'trajectory_time': trajectory_time_used}
    for idx in range(pressure.shape[1]):
        npz_dict[f'p{idx+1}'] = pressure[:, idx]
else:
    if "smooth" in file_name:
        npz_dict = {'images': images[4500:], 'trajectory_time': trajectory_time_used[4500:]}
        for idx in range(pressure.shape[1]):
            npz_dict[f'p{idx+1}'] = pressure[4500:, idx]
    else:
        npz_dict = {'images': images, 'trajectory_time': trajectory_time_used}
        for idx in range(pressure.shape[1]):
            npz_dict[f'p{idx+1}'] = pressure[:, idx]



out_file = f"data/scr_match2/{file_name}_processed.npz"
np.savez_compressed(out_file, **npz_dict)
print(f"Processed dataset saved to {out_file}")




# Generate static dataset from normal, step dataset

In [ ]:
datasets = {
    "scr_match_1seg": f"scr_match2/1segment_step_input_rand_compressed_processed.npz",
    "scr_match_2seg": f"scr_match2/2segments_step_input_rand_18s_compressed_processed.npz",
}

dataset_path = f"data/{datasets['scr_match_1seg']}"

# dataset="soft_finger/soft_finger_beam_bent_32x32_ss4_rgb.npz"
# dataset="double_pendulum/double_pendulum_32x32_rgb_10k.npz"

dataset=np.load(dataset_path)

#

## Get static indices

In [ ]:
import matplotlib.pyplot as plt

# For standalone interactive window in Jupyter, use the qt backend
%matplotlib qt

plt.figure()  # Open a new figure window


plt.plot(dataset['p1'])
plt.plot(dataset['p2'])
if "2seg" in dataset_path:
    plt.plot(dataset['p3'])
    plt.plot(dataset['p4'])

# 2 segments
if "2seg" in dataset_path:
    n_space = int((18)*50.0)
    n_offset = 1782
else:
    n_space = int((4)*50.0+1)
    n_offset = 2302

# Plot markers with spacing of n_space
if "2seg" in dataset_path:
    for idx, p in enumerate(['p1','p2','p3','p4']):
        static_indices = np.arange(n_offset, len(dataset[p]), n_space)
        ydata = dataset[p][static_indices]

    plt.scatter(static_indices, ydata, color="r")  # Base plot without markers
else:
    for idx, p in enumerate(['p1','p2']):
        static_indices = np.arange(n_offset, len(dataset[p]), n_space)
        ydata = dataset[p][static_indices]
        plt.scatter(static_indices, ydata, color="r")  # Base plot without markers
plt.show()  # Open the plot window to enable zoom/pan


In [ ]:
n_window = 50

min_mse_image_indices = []
mean_pressures = []

import numpy as np
import matplotlib.pyplot as plt

for idx_static_point, idx_value in enumerate(static_indices):
    idx_start = idx_value - n_window
    idx_end = idx_value

    # Stack images and pressure data in window before the static point
    images_static_point = dataset['images'][idx_start:idx_end]
    mean_p1 = np.mean(dataset['p1'][idx_start:idx_end])
    mean_p2 = np.mean(dataset['p2'][idx_start:idx_end])
    if "2seg" in dataset_path:
        mean_p3 = np.mean(dataset['p3'][idx_start:idx_end])
        mean_p4 = np.mean(dataset['p4'][idx_start:idx_end])
        mean_pressures.append([mean_p1, mean_p2, mean_p3, mean_p4])
    else:
        mean_pressures.append([mean_p1, mean_p2])

    # Compute the average image
    mean_image = np.mean(images_static_point, axis=0)

    # Compute MSE for each image vs the mean image
    mses = ((images_static_point - mean_image)**2).mean(axis=(1,2))

    # Find the index with the minimum MSE (relative to window start)
    min_mse_idx_in_window = np.argmin(mses)
    # Convert to absolute dataset index
    min_mse_abs_idx = idx_start + min_mse_idx_in_window
    min_mse_image_indices.append(min_mse_abs_idx)

    # Optional: plot for each static point
    # plt.figure()
    # plt.plot(range(len(mses)), mses, marker='o', label='MSE')
    # plt.scatter(min_mse_idx_in_window, mses[min_mse_idx_in_window], color='red', zorder=5, label='Min MSE')
    # plt.xlabel("Image Index (in window)")
    # plt.ylabel("MSE vs Average Image")
    # plt.title(f"MSE Window {idx_static_point}")
    # plt.grid(True)
    # plt.legend()
    # plt.show()

static_dataset_pressures = np.array(mean_pressures)  # Shape: (num_static_points, 4)
static_dataset_images = dataset['images'][min_mse_image_indices]  # Shape: (num_static_points,)
# min_mse_image_indices now holds the dataset indices of the images with lowest MSE for each static point
# mean_pressures is an array of [mean_p1, mean_p2, mean_p3, mean_p4] for each static point

In [ ]:
# Save results to npz file
npz_dict = {'images': static_dataset_images, }
for idx in range(static_dataset_pressures.shape[1]):
    npz_dict[f'p{idx+1}'] = static_dataset_pressures[:, idx]
out_file = f"data/scr_match2/{file_name}_static_processed.npz"
np.savez_compressed(out_file, **npz_dict)
print(f"Processed dataset saved to {out_file}")

In [ ]:
datasets = {
    "scr_match_1seg": f"scr_match2/1segment_step_input_rand_compressed_processed.npz",
    "scr_match_2seg": f"scr_match2/2segments_step_input_rand_18s_compressed_static_processed.npz",
}

dataset_path = f"data/{datasets['scr_match_2seg']}"

# dataset="soft_finger/soft_finger_beam_bent_32x32_ss4_rgb.npz"
# dataset="double_pendulum/double_pendulum_32x32_rgb_10k.npz"

dataset=np.load(dataset_path)

#